# Hard Case 분석
### 핵심 질문: 오차가 큰 케이스에는 어떤 공통 패턴이 있는가?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Arc
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import os
os.makedirs('figures', exist_ok=True)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../oof_v8_simple.csv')
df['pred_dist'] = np.sqrt(df['pred_dx']**2 + df['pred_dy']**2)
hard = df[df['error_dist'] > 30].copy()
easy = df[df['error_dist'] <= 30].copy()
print(f'전체: {len(df)}  |  hard(>30m): {len(hard)} ({len(hard)/len(df)*100:.1f}%)  |  easy: {len(easy)}')
print(f'전체 평균 오차: {df["error_dist"].mean():.2f}m, 중앙값: {df["error_dist"].median():.2f}m')

In [ ]:
def draw_pitch(ax):
    ax.set_facecolor('#3d8c40')
    for x in [[0,0],[0,105],[105,105],[105,0]]:
        ax.plot([x[0],x[1] if x[0]==x[1] else x[1]],
                [0 if x[0]==0 and x[1]==0 else 68 if x[0]==0 and x[1]==105 else 0,
                 68 if x[0]==0 and x[1]==0 else 68 if x[0]==0 and x[1]==105 else 0], 'w-')
    ax.plot([0,0],[0,68],'w-',lw=2); ax.plot([0,105],[68,68],'w-',lw=2)
    ax.plot([105,105],[68,0],'w-',lw=2); ax.plot([105,0],[0,0],'w-',lw=2)
    ax.plot([52.5,52.5],[0,68],'w-',lw=1.5)
    ax.add_patch(plt.Circle((52.5,34),9.15,color='w',fill=False,lw=1.5))
    ax.plot([0,16.5],[13.84,13.84],'w-',lw=1.5); ax.plot([16.5,16.5],[13.84,54.16],'w-',lw=1.5)
    ax.plot([0,16.5],[54.16,54.16],'w-',lw=1.5)
    ax.plot([105,88.5],[13.84,13.84],'w-',lw=1.5); ax.plot([88.5,88.5],[13.84,54.16],'w-',lw=1.5)
    ax.plot([88.5,105],[54.16,54.16],'w-',lw=1.5)
    ax.set_xlim(-3,108); ax.set_ylim(-3,71); ax.set_aspect('equal'); ax.axis('off')
    return ax

## 시각화 1. Hard Case 시작점 공간 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('예측 오차 크기별 패스 시작점 분포', fontsize=15, fontweight='bold')

ax1 = draw_pitch(axes[0])
hb1 = ax1.hexbin(easy['start_x'], easy['start_y'], gridsize=14, cmap='Blues', alpha=0.75, mincnt=1)
plt.colorbar(hb1, ax=ax1, label='Count', shrink=0.8)
ax1.axvline(easy['start_x'].mean(), color='cyan', linestyle='--', lw=2,
            label=f'평균 x: {easy["start_x"].mean():.1f}m')
ax1.set_title(f'쉬운 케이스 (오차 ≤30m, n={len(easy):,})', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10, framealpha=0.8)

ax2 = draw_pitch(axes[1])
hb2 = ax2.hexbin(hard['start_x'], hard['start_y'], gridsize=14, cmap='Reds', alpha=0.75, mincnt=1)
plt.colorbar(hb2, ax=ax2, label='Count', shrink=0.8)
ax2.axvline(hard['start_x'].mean(), color='yellow', linestyle='--', lw=2,
            label=f'평균 x: {hard["start_x"].mean():.1f}m')
ax2.set_title(f'어려운 케이스 (오차 >30m, n={len(hard):,})', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=10, framealpha=0.8)

plt.tight_layout()
plt.savefig('figures/hard_case_start_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 시각화 2. 패스 거리와 오차의 관계 — Under-prediction 패턴

In [ ]:
bins = [0, 10, 20, 30, 50, 200]
labels = ['0-10m', '10-20m', '20-30m', '30-50m', '50m+']
df['dist_bin'] = pd.cut(df['target_dist'], bins=bins, labels=labels)

grouped = df.groupby('dist_bin', observed=False).agg(
    actual_mean=('target_dist', 'mean'),
    pred_mean=('pred_dist', 'mean'),
    error_mean=('error_dist', 'mean'),
    count=('error_dist', 'count')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('패스 거리와 예측 오차: Under-prediction 패턴 분석', fontsize=14, fontweight='bold')

# 왼쪽: 실제 vs 예측 거리 (구간별)
ax1 = axes[0]
x = np.arange(len(grouped))
w = 0.35
b1 = ax1.bar(x - w/2, grouped['actual_mean'], w, label='실제 패스 거리',
             color='#3498DB', edgecolor='black', linewidth=0.7, alpha=0.85)
b2 = ax1.bar(x + w/2, grouped['pred_mean'], w, label='예측 패스 거리',
             color='#E74C3C', edgecolor='black', linewidth=0.7, alpha=0.85)

# under-prediction 화살표
for i, row in grouped.iterrows():
    gap = row['actual_mean'] - row['pred_mean']
    if gap > 3:
        ax1.annotate('', xy=(i+w/2, row['pred_mean']+1), xytext=(i-w/2, row['actual_mean']-1),
                     arrowprops=dict(arrowstyle='->', color='#7D3C98', lw=1.5))
        ax1.text(i+0.05, (row['actual_mean']+row['pred_mean'])/2,
                 f'-{gap:.0f}m', color='#7D3C98', fontsize=9, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(
    [f'{row["dist_bin"]}\n(n={row["count"]:,})' for _, row in grouped.iterrows()],
    fontsize=9
)
ax1.set_ylabel('패스 거리 (m)', fontsize=11)
ax1.set_title('거리 구간별 실제 vs 예측 패스 거리', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)

# 오른쪽: 실제 거리 vs 오차 산점도
ax2 = axes[1]
sc = ax2.scatter(df['target_dist'], df['error_dist'],
                 alpha=0.12, s=6,
                 c=df['start_x'], cmap='RdYlGn', vmin=0, vmax=105)
plt.colorbar(sc, ax=ax2, label='시작 x좌표 (수비←→공격)', shrink=0.8)

ax2.plot(grouped['actual_mean'], grouped['error_mean'],
         'k-o', linewidth=2.5, markersize=8, label='구간 평균 오차', zorder=5)
for _, row in grouped.iterrows():
    ax2.text(row['actual_mean']+0.5, row['error_mean']+0.5,
             f'{row["error_mean"]:.1f}m', fontsize=9, fontweight='bold')

ax2.set_xlabel('실제 패스 거리 (m)', fontsize=11)
ax2.set_ylabel('예측 오차 (m)', fontsize=11)
ax2.set_title('실제 패스 거리 vs 예측 오차\n(색상: 시작점 위치)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('figures/underprediction_pattern.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print('[Under-prediction 요약]')
for _, row in grouped.iterrows():
    gap = row['actual_mean'] - row['pred_mean']
    print(f"  {row['dist_bin']}: 실제 {row['actual_mean']:.1f}m → 예측 {row['pred_mean']:.1f}m (차이 {gap:.1f}m)")

## 시각화 3. 수비 지역이 왜 어려운가 — 롱패스 비율과 오차

In [ ]:
zone_bins = [0, 35, 70, 105]
zone_labels = ['수비 (x<35)', '중원 (35~70)', '공격 (x≥70)']
df['zone'] = pd.cut(df['start_x'], bins=zone_bins, labels=zone_labels)

zone_stats = df.groupby('zone', observed=True).agg(
    mean_error=('error_dist', 'mean'),
    mean_target_dist=('target_dist', 'mean'),
    long_pass_rate=('target_dist', lambda x: (x > 30).mean() * 100),
    count=('error_dist', 'count')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('필드 존별 분석: 수비 지역이 왜 어려운가', fontsize=14, fontweight='bold')

zone_colors = ['#E74C3C', '#F39C12', '#27AE60']

# 왼쪽: 존별 평균 오차
ax1 = axes[0]
bars = ax1.bar(range(3), zone_stats['mean_error'], color=zone_colors,
               edgecolor='black', linewidth=0.7, alpha=0.85)
ax1.set_xticks(range(3))
ax1.set_xticklabels(
    [f'{row["zone"]}\n(n={row["count"]:,})' for _, row in zone_stats.iterrows()],
    fontsize=9
)
ax1.set_ylabel('평균 예측 오차 (m)', fontsize=11)
ax1.set_title('존별 평균 예측 오차', fontsize=12, fontweight='bold')
for bar, val in zip(bars, zone_stats['mean_error']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{val:.1f}m', ha='center', fontsize=11, fontweight='bold')

# 가운데: 존별 평균 패스 거리
ax2 = axes[1]
bars2 = ax2.bar(range(3), zone_stats['mean_target_dist'], color=zone_colors,
                edgecolor='black', linewidth=0.7, alpha=0.85)
ax2.set_xticks(range(3))
ax2.set_xticklabels(
    [f'{row["zone"]}\n(n={row["count"]:,})' for _, row in zone_stats.iterrows()],
    fontsize=9
)
ax2.set_ylabel('평균 패스 거리 (m)', fontsize=11)
ax2.set_title('존별 평균 패스 거리', fontsize=12, fontweight='bold')
for bar, val in zip(bars2, zone_stats['mean_target_dist']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{val:.1f}m', ha='center', fontsize=11, fontweight='bold')

# 오른쪽: 존별 롱패스 비율
ax3 = axes[2]
bars3 = ax3.bar(range(3), zone_stats['long_pass_rate'], color=zone_colors,
                edgecolor='black', linewidth=0.7, alpha=0.85)
ax3.set_xticks(range(3))
ax3.set_xticklabels(
    [f'{row["zone"]}\n(n={row["count"]:,})' for _, row in zone_stats.iterrows()],
    fontsize=9
)
ax3.set_ylabel('롱패스 비율 (%, 거리>30m)', fontsize=11)
ax3.set_title('존별 롱패스 비율', fontsize=12, fontweight='bold')
for bar, val in zip(bars3, zone_stats['long_pass_rate']):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/zone_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(zone_stats[['zone','mean_error','mean_target_dist','long_pass_rate']].to_string())

## 시각화 4. Worst 20 케이스 — 실제 vs 예측 패스

In [ ]:
worst = df.nlargest(20, 'error_dist')

fig, ax = plt.subplots(figsize=(14, 9))
draw_pitch(ax)

for _, row in worst.iterrows():
    ax.plot(row['start_x'], row['start_y'], 'ko', markersize=7, zorder=5)
    ax.annotate('', xy=(row['true_end_x'], row['true_end_y']),
                xytext=(row['start_x'], row['start_y']),
                arrowprops=dict(arrowstyle='->', color='#3498DB', lw=2, alpha=0.85))
    ax.annotate('', xy=(row['pred_end_x'], row['pred_end_y']),
                xytext=(row['start_x'], row['start_y']),
                arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2,
                                alpha=0.85, linestyle='dashed'))

ax.plot([], [], 'b-', lw=2.5, label='실제 패스')
ax.plot([], [], 'r--', lw=2.5, label='예측 패스')
ax.plot([], [], 'ko', markersize=7, label='패스 시작점')
ax.legend(loc='upper right', fontsize=12, framealpha=0.9)

ax.set_title(
    f'예측 오차 Top 20\n'
    f'평균 시작 x: {worst["start_x"].mean():.1f}m  |  '
    f'실제 평균 거리: {worst["target_dist"].mean():.1f}m  |  '
    f'예측 평균 거리: {worst["pred_dist"].mean():.1f}m  |  '
    f'평균 오차: {worst["error_dist"].mean():.1f}m',
    fontsize=12, fontweight='bold'
)

plt.tight_layout()
plt.savefig('figures/worst_20_cases.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Worst 20 요약:')
print(f'  시작 x 평균: {worst["start_x"].mean():.1f}m (전체: {df["start_x"].mean():.1f}m)')
print(f'  실제 패스 거리: {worst["target_dist"].mean():.1f}m')
print(f'  예측 패스 거리: {worst["pred_dist"].mean():.1f}m')
print(f'  오차 범위: {worst["error_dist"].min():.1f}m ~ {worst["error_dist"].max():.1f}m')

## 종합 요약

In [ ]:
print('=' * 65)
print('Hard Case 분석 요약')
print('=' * 65)

print(f'\n[전체]')
print(f'  평균 오차: {df["error_dist"].mean():.2f}m  |  중앙값: {df["error_dist"].median():.2f}m')

print(f'\n[Hard Case 특징]')
print(f'  수: {len(hard):,}개 ({len(hard)/len(df)*100:.1f}%)')
print(f'  평균 시작 x: {hard["start_x"].mean():.1f}m  (전체: {df["start_x"].mean():.1f}m)')
print(f'  실제 패스 거리: {hard["target_dist"].mean():.1f}m')
print(f'  예측 패스 거리: {hard["pred_dist"].mean():.1f}m  (→ {hard["target_dist"].mean()/hard["pred_dist"].mean():.1f}배 짧게 예측)')

print(f'\n[수비 vs 공격 지역 비교]')
defense = df[df['start_x'] < 35]
attack = df[df['start_x'] >= 70]
print(f'  수비: 평균 오차 {defense["error_dist"].mean():.1f}m  |  롱패스 비율 {(defense["target_dist"]>30).mean()*100:.1f}%')
print(f'  공격: 평균 오차 {attack["error_dist"].mean():.1f}m   |  롱패스 비율 {(attack["target_dist"]>30).mean()*100:.1f}%')

print(f'\n[결론]')
print(f'  수비 지역은 롱패스 비율이 {(defense["target_dist"]>30).mean()*100:.1f}%로 높음')
print(f'  MSE 손실은 극단값보다 평균에 가까운 예측이 유리 → 롱패스를 항상 짧게 예측')
print(f'  두 조건이 겹치는 "수비 지역 + 롱패스" 조합이 Hard Case의 핵심')